# Phase 5: Data Preprocessing & ML Pipeline Preparation
**Objective:** To convert the validated geotechnical telemetry dataset into a machine-learning-ready matrix by engineering categorical encodings, feature normalization, outlier control, and class balancing strategies.

---

### 🎯 Core Engineering Objectives:
1. Preserve geotechnical realism while handling incomplete excavation records.
2. Convert categorical site descriptors into numerical tensors for ML ingestion.
3. Normalize heterogeneous IoT telemetry scales into a unified feature space.
4. Address severe target imbalance to prevent predictive bias toward failure states.
5. Produce a finalized tensor dataset for predictive excavation safety modeling.

### Phase 5.1: Preprocessing Initialization
**Objective:** To initialize the environment, load the raw dataset, and architect a strict segregation between continuous physical measurements and categorical site descriptions.

In [26]:
# =========================================================
# PHASE 5.1 — PREPROCESSING INITIALIZATION
# =========================================================
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Professional styling
warnings.filterwarnings('ignore')

# 1. Load Data
df = pd.read_csv('../data/raw/excavation_risk_dataset.csv')
TARGET_VAR = 'Risk_Level'

# 2. Define Feature Types
categorical_features = ['Soil_Type', 'Retaining_Wall_Type', 'Support_System']
continuous_features = [
    'Excavation_Depth_m', 'Shear_Strength_kPa', 'Bearing_Capacity_kPa', 
    'Groundwater_Level_m', 'Deformation_mm', 'Rainfall_mm_day', 
    'Temperature_C', 'Seismic_Activity', 'Ground_Settlement_mm', 
    'Wall_Displacement_mm', 'Pore_Water_Pressure_kPa', 'Strain_Gauge', 
    'Soil_Moisture_%'
]

print(f"✅ Preprocessing Environment Initialized. Raw Matrix Shape: {df.shape}")

✅ Preprocessing Environment Initialized. Raw Matrix Shape: (1000, 17)


#### 📝 Phase 5.1 Inference: Pipeline Segregation
* By explicitly isolating `categorical_features` from `continuous_features`, we establish a defensive architecture. This guarantees that downstream mathematical algorithms (like Standard Scaling) will exclusively target physical measurements, preventing the corruption of nominal categories.

### Phase 5.2: Engineering-Based Imputation
**Objective:** To address missing values in the dataset using Civil Engineering domain knowledge rather than destructive algorithmic deletion.

In [27]:
# =========================================================
# PHASE 5.2 — ENGINEERING-BASED IMPUTATION
# =========================================================
# Impute missing structural supports as an active engineering state
df['Support_System'] = df['Support_System'].fillna('None (Open Cut)')

print(f"✅ Imputation Complete. Remaining Nulls: {df.isnull().sum().sum()}")

✅ Imputation Complete. Remaining Nulls: 0


#### 📝 Phase 5.2 Inference: Domain-Aware Imputation
* Standard Data Science practice often dictates dropping rows with missing values (`dropna()`). However, in geotechnical engineering, a missing structural support does not indicate data corruption; it signifies an **Open-Cut (unsupported) excavation**. Imputing `'None (Open Cut)'` preserves this critical physical reality.

### Phase 5.3: Categorical Feature Encoding
**Objective:** To convert textual site descriptors (e.g., "Clay", "Sheet Pile") into a mathematical format the machine learning algorithm can process.

In [28]:
# =========================================================
# PHASE 5.3 — CATEGORICAL FEATURE ENCODING
# =========================================================
# Apply One-Hot Encoding to prevent ordinal mathematical bias
df_processed = pd.get_dummies(df, columns=categorical_features, drop_first=False)

print(f"✅ Categorical Features Encoded via OHE. New Shape: {df_processed.shape}")

✅ Categorical Features Encoded via OHE. New Shape: (1000, 24)


#### 📝 Phase 5.3 Inference: Physics Preservation via OHE
* **The Ordinal Trap:** We strictly avoided Ordinal/Label Encoders. Assigning (0, 1, 2) to soil types introduces false mathematical sequences (forcing the algorithm to assume $Clay < Rock < Silt$). 
* **The OHE Solution:** **One-Hot Encoding (OHE)** creates isolated binary tensors, ensuring the Machine Learning model treats each soil profile and support system as an independent, non-overlapping engineering state.

### Phase 5.4: Feature Matrix Segregation
**Objective:** To isolate the predictive input vectors ($X$) from the target output variable ($y$).

In [29]:
# =========================================================
# PHASE 5.4 — FEATURE MATRIX SEGREGATION
# =========================================================
# Isolate predictive vectors from the target
X = df_processed.drop(TARGET_VAR, axis=1)
y = df_processed[TARGET_VAR]

print(f"✅ Feature Segregation Complete.")
print(f"   -> Input Matrix (X): {X.shape}")
print(f"   -> Target Vector (y): {y.shape}")

✅ Feature Segregation Complete.
   -> Input Matrix (X): (1000, 23)
   -> Target Vector (y): (1000,)


#### 📝 Phase 5.4 Inference: Structural Formatting
* The dataset is now structurally formatted for supervised machine learning, separating the independent variables (telemetry and site profile) from the dependent variable (Risk Level).

### Phase 5.5: Train / Test Segregation (Anti-Leakage Protocol)
**Objective:** To strictly divide the dataset into Training and Testing subsets *before* any statistical scaling or balancing occurs, guaranteeing the integrity of our final evaluation.

In [30]:
# =========================================================
# PHASE 5.5 — TRAIN / TEST SEGREGATION
# =========================================================
# CRITICAL: Segregation MUST occur BEFORE scaling to prevent Data Leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y  # Maintains realistic class distributions in both sets
)

print("✅ Anti-Leakage Partitioning Complete.")
print(f"   -> Training Set: {X_train.shape} | Testing Set: {X_test.shape}")

✅ Anti-Leakage Partitioning Complete.
   -> Training Set: (800, 23) | Testing Set: (200, 23)


#### 📝 Phase 5.5 Inference: The Data Leakage Paradigm
* **Architectural Shift:** We executed Train/Test Segregation *before* Normalization. If standard scaling is applied to the global dataset first, the scaler calculates its mean using the "unseen" Test Data. This mathematical detail "leaks" into the training data, artificially inflating model accuracy. By splitting now, we ensure a mathematically valid evaluation.

### Phase 5.6: Telemetry Normalization
**Objective:** To standardize the disparate scales of our continuous IoT sensors so that features with massive units (like PWP in kPa) do not overpower features with smaller units (like Displacement in mm).

In [31]:
# =========================================================
# PHASE 5.6 — TELEMETRY NORMALIZATION
# =========================================================
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit on Training Data ONLY. Transform both. 
# Apply EXCLUSIVELY to continuous physical measurements.
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

print("✅ Selective Telemetry Normalization Complete.")

✅ Selective Telemetry Normalization Complete.


#### 📝 Phase 5.6 Inference: Selective Normalization
* **Targeted Scaling:** Scaling was limited strictly to continuous telemetry. Had we indiscriminately scaled the entire matrix, the crisp binary logic (0 and 1) of our OHE columns would have been destroyed into decimal artifacts.
* **Fit vs. Transform:** We fit the scaler exclusively on `X_train`, and merely transformed `X_test`. This mimics production reality, where the model must process live sensor data based on historical training baselines.

### Phase 5.7: Outlier Detection (IQR Method)
**Objective:** To mathematically scan the training data for statistical outliers in the structural response telemetry.

In [32]:
# =========================================================
# PHASE 5.7 — OUTLIER DETECTION (IQR METHOD)
# =========================================================
def detect_outliers_iqr(df_in, features):
    outlier_count = 0
    for col in features:
        Q1 = df_in[col].quantile(0.25)
        Q3 = df_in[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = Q3 + 1.5 * IQR
        outliers = df_in[df_in[col] > upper_bound].shape[0]
        outlier_count += outliers
    return outlier_count

outliers_detected = detect_outliers_iqr(X_train_scaled, continuous_features)
print(f"⚠️ IQR Scan: Detected {outliers_detected} statistical outliers in training telemetry.")

⚠️ IQR Scan: Detected 0 statistical outliers in training telemetry.


#### 📝 Phase 5.7 Inference: The SHM Outlier Paradigm
* While the IQR method successfully flagged statistical outliers, **no clipping or dropping was applied**. In commercial data science, outliers are noise. In Civil Engineering Structural Health Monitoring (SHM), extreme outliers represent actual catastrophic physical failures (e.g., severe wall displacement). Retaining them is vital for the predictive model.

### Phase 5.8: Target Balancing using SMOTE
**Objective:** To rectify the severe `Risk_Level` imbalance (~70% Danger states) exclusively within the training environment to prevent predictive failure bias.

In [33]:
# =========================================================
# PHASE 5.8 — TARGET BALANCING USING SMOTE
# =========================================================
# SMOTE is applied STRICTLY to the Training Data
smote = SMOTE(random_state=42)
X_train_final, y_train_final = smote.fit_resample(X_train_scaled, y_train)

print(f"✅ SMOTE Applied (Training Set Only). New Balance:\n{y_train_final.value_counts()}")

✅ SMOTE Applied (Training Set Only). New Balance:
Risk_Level
2    647
1    647
0    647
Name: count, dtype: int64


#### 📝 Phase 5.8 Inference: Training Reality vs. Testing Reality
* **The Real-World Anchor:** It is a strict ML rule that SMOTE must *only* be applied to the Training Set. By equalizing the training data, the algorithm learns all risk states equally. By leaving the Testing Set imbalanced, we ensure our final evaluation reflects the genuine, skewed reality of a construction site.

### Phase 5.9: Finalized ML Dataset Export
**Objective:** To package and export the mathematically validated arrays and the trained scaler for use in algorithmic training and eventual backend deployment.

In [34]:
# =========================================================
# PHASE 5.9 — FINALIZED ML DATASET EXPORT
# =========================================================
import joblib

# Exporting the trained scaler for real-time MERN backend use
# joblib.dump(scaler, '../../data/processed/telemetry_scaler.pkl')

processed_train = X_train_final.copy()
processed_train[TARGET_VAR] = y_train_final

processed_test = X_test_scaled.copy()
processed_test[TARGET_VAR] = y_test

processed_train.to_csv('../data/processed/X_train_processed.csv', index=False)
processed_test.to_csv('../data/processed/X_test_processed.csv', index=False)

print("✅ Phase 5 Complete: Pipeline Arrays and Scaler ready for Algorithm ingestion.")

✅ Phase 5 Complete: Pipeline Arrays and Scaler ready for Algorithm ingestion.


#### 📝 Phase 5.9 Inference: Production Readiness
* We export the fully processed matrices for the next notebook (`03_model_training`). 
* More importantly, we export the `telemetry_scaler.pkl`. When the MERN stack dashboard streams live, unseen IoT data, the Node.js backend will use this exact file to scale the real-time data identically to how the model was trained, ensuring continuous predictive stability.

*End of Preprocessing Notebook. Proceed to Model Training.*